In [2]:
import torch
import torch.nn as nn
import h5py
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer
from statsmodels.tsa.stattools import grangercausalitytests
from torch_geometric.utils import from_scipy_sparse_matrix, add_self_loops
from scipy.sparse import coo_matrix
import time
import math
import random
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm
import json
import evaluate as hf_evaluate
import os

# ==================================================================================
# CONFIGURATION
# ==================================================================================

# --- UPDATED PATHS ---
H5_FILE_PATH = "/home/poorna/data/eeg_dataset_1400_multilabel.h5"
LOCAL_MODEL_PATH = "/home/poorna/models/bert-base-uncased"
OBJECT_MAPPING_FILE = "/home/poorna/data/object_id_to_name_blip.json"

TRAIN_PCT, VAL_PCT = 0.8, 0.1
BATCH_SIZE = 16
EPOCHS = 30 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)
PAD_ID = tokenizer.pad_token_id
SOS_ID = tokenizer.cls_token_id
EOS_ID = tokenizer.sep_token_id
TEXT_VOCAB_SIZE = tokenizer.vocab_size

# --- UPDATED DIMENSIONS ---
NUM_COLORS = 9       # Black, Blue, Brown, Green, Grey, Orange, Red, White, Yellow
NUM_OBJECTS = 6      # Animal, Building, Food, Nature, Person, Vehicle
# NUM_CATEGORIES removed as it does not exist in the 1400 dataset

/home/poorna/venvs/torch/lib64/python3.11/site-packages/sklearn/utils/_param_validation.py:14: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.3)
  from scipy.sparse import csr_matrix, issparse


Using device: cuda


In [3]:
# ==================================================================================
# GRANGER CAUSALITY
# ==================================================================================
def create_granger_causality_matrix(eeg_batch):
    eeg_sample = eeg_batch[0].cpu().numpy().T
    num_channels = eeg_sample.shape[1]
    causality_matrix = np.zeros((num_channels, num_channels))

    for i in range(num_channels):
        for j in range(num_channels):
            if i == j:
                continue
            ts_i = eeg_sample[:, i]
            ts_j = eeg_sample[:, j]
            min_len = 20
            if len(ts_i) < min_len or len(ts_j) < min_len:
                causality_matrix[i, j] = 0.0
                continue
            data = np.vstack([ts_j, ts_i]).T
            try:
                current_maxlag = min(5, len(data)//2 - 2)
                if current_maxlag < 1:
                    causality_matrix[i, j] = 0.0
                    continue
                results = grangercausalitytests(data, maxlag=current_maxlag, verbose=False)
                p_value = results[current_maxlag][0]['ssr_ftest'][1]
                if p_value < 0.05:
                    causality_matrix[i, j] = 1.0
            except Exception as e:
                causality_matrix[i, j] = 0.0

    adj_matrix = coo_matrix(causality_matrix)
    edge_index, edge_attr = from_scipy_sparse_matrix(adj_matrix)

    if edge_attr is None:
        edge_attr = torch.tensor([], dtype=torch.float)
    elif edge_attr.ndim == 0:
        edge_attr = edge_attr.unsqueeze(0)

    return edge_index.to(torch.long), edge_attr.to(torch.float)

In [4]:
# ==================================================================================
# DATASET
# ==================================================================================
class EEGMetaTextH5Dataset(Dataset):
    def __init__(self, h5_path):
        self.h5_path = h5_path
        self.h5_file = None
        with h5py.File(self.h5_path, 'r') as f:
            self.n_samples = f['eeg'].shape[0]

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        if self.h5_file is None:
            self.h5_file = h5py.File(self.h5_path, 'r')

        eeg = torch.from_numpy(self.h5_file['eeg'][idx].astype(np.float32))
        # Metadata is now [9 colors, 6 objects]
        meta = torch.from_numpy(self.h5_file['metadata'][idx].astype(np.float32))
        text = torch.from_numpy(self.h5_file['input_ids'][idx].astype(np.int64))

        return eeg, meta, text

def collate_multimodal_batch(batch):
    eeg_list, meta_list, text_list = [], [], []
    for eeg, meta, txt in batch:
        eeg_list.append(eeg)
        meta_list.append(meta)
        text_list.append(txt)

    eeg_batch = torch.stack(eeg_list, dim=0)
    meta_batch = torch.stack(meta_list, dim=0)
    text_padded = pad_sequence(text_list, batch_first=True, padding_value=PAD_ID)

    return eeg_batch.float(), meta_batch.float(), text_padded

In [5]:
# ==================================================================================
# MODEL COMPONENTS
# ==================================================================================

class SpatioTemporalEEGEncoder(nn.Module):
    def __init__(self, num_channels=62, enc_hidden=256, num_layers=2, dropout=0.2):
        super().__init__()
        self.num_channels = num_channels
        self.gcn1 = GCNConv(num_channels, enc_hidden)
        self.gcn2 = GCNConv(enc_hidden, enc_hidden)
        self.rnn = nn.GRU(enc_hidden, enc_hidden, num_layers,
                          bidirectional=True, dropout=dropout if num_layers > 1 else 0,
                          batch_first=True)
        self.dropout = nn.Dropout(dropout)
        print(f"Encoder RNN input size: {enc_hidden}")

    def forward(self, eeg, edge_index, edge_attr):
        batch_size = eeg.shape[0]
        num_timesteps = eeg.shape[2]

        batch_edge_index = edge_index.repeat(1, batch_size)
        batch_edge_attr = edge_attr.repeat(batch_size)
        batch_offset = torch.arange(batch_size, device=eeg.device) * self.num_channels
        batch_edge_index = batch_edge_index + batch_offset.repeat_interleave(edge_index.shape[1])

        eeg_reshaped = eeg.permute(0, 2, 1).reshape(-1, self.num_channels)

        x = F.relu(self.gcn1(eeg_reshaped, batch_edge_index, batch_edge_attr))
        x = self.dropout(x)
        x = F.relu(self.gcn2(x, batch_edge_index, batch_edge_attr))

        temporal_features = x.reshape(batch_size, num_timesteps, -1)
        encoder_outputs, encoder_hidden = self.rnn(temporal_features)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)

        return encoder_outputs, encoder_hidden


class LuongAttention(nn.Module):
    def __init__(self, enc_dim, dec_dim):
        super().__init__()
        self.attn = nn.Linear(enc_dim, dec_dim)

    def forward(self, decoder_hidden, encoder_outputs):
        src_len = encoder_outputs.shape[0]
        attn_energies = self.attn(encoder_outputs)
        scores = torch.bmm(decoder_hidden.permute(1, 0, 2), attn_energies.permute(1, 2, 0))
        attn_weights = F.softmax(scores, dim=2)
        context = torch.bmm(attn_weights, encoder_outputs.permute(1, 0, 2))
        return context, attn_weights.squeeze(1)


class MetadataEncoder(nn.Module):
    """Updated for Multi-Label Color (9) + Objects (6), No Categories"""
    def __init__(self, num_colors, num_objects,
                 color_feature_dim=32, object_feature_dim=32):
        super().__init__()
        
        # Changed from Embedding to Linear because input is now multi-hot vector
        self.color_processor = nn.Sequential(
            nn.Linear(num_colors, 64),
            nn.ReLU(),
            nn.Linear(64, color_feature_dim)
        )

        self.object_processor = nn.Sequential(
            nn.Linear(num_objects, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, object_feature_dim)
        )

        self.output_dim = color_feature_dim + object_feature_dim
        print(f"MetadataEncoder output dimension: {self.output_dim}")

    def forward(self, metadata):
        # Split: [batch, 9+6] -> [batch, 9] colors, [batch, 6] objects
        color_input = metadata[:, :NUM_COLORS].float()
        object_input = metadata[:, NUM_COLORS:].float()

        color_vec = self.color_processor(color_input)
        object_vec = self.object_processor(object_input)

        combined_features = torch.cat([color_vec, object_vec], dim=1)
        return combined_features


class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, enc_hidden, dec_hidden, meta_features_dim, num_layers, pad_id, dropout):
        super().__init__()
        self.vocab_size = vocab_size
        self.dec_hidden = dec_hidden
        self.num_layers = num_layers
        enc_dim = enc_hidden * 2

        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.attention = LuongAttention(enc_dim, dec_hidden)

        self.rnn_input_dim = emb_dim + enc_dim + meta_features_dim + enc_dim
        print(f"Decoder RNN input dimension: {self.rnn_input_dim}")
        self.rnn = nn.GRU(self.rnn_input_dim, dec_hidden, num_layers, dropout=dropout if num_layers > 1 else 0)

        self.fc_out = nn.Linear(dec_hidden, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.bridge = nn.Linear(enc_dim, dec_hidden)

    def init_hidden(self, encoder_hidden):
        hidden = encoder_hidden.view(self.num_layers, 2, encoder_hidden.size(1), -1)
        last_layer_hidden = hidden[-1]
        encoder_hidden_cat = torch.cat((last_layer_hidden[0], last_layer_hidden[1]), dim=1)
        bridged_hidden = torch.tanh(self.bridge(encoder_hidden_cat))
        decoder_initial_hidden = bridged_hidden.unsqueeze(0).repeat(self.num_layers, 1, 1)
        return decoder_initial_hidden

    def forward(self, token, decoder_hidden, encoder_outputs, meta_features, global_eeg_context):
        token = token.unsqueeze(0)
        embedded = self.dropout(self.embedding(token))
        context, attn_weights = self.attention(decoder_hidden[-1].unsqueeze(0), encoder_outputs)
        
        meta_features_unsqueezed = meta_features.unsqueeze(0)
        global_eeg_context_unsqueezed = global_eeg_context.unsqueeze(0)
        context_permuted = context.permute(1, 0, 2)

        rnn_input = torch.cat((
            embedded,
            context_permuted,
            meta_features_unsqueezed,
            global_eeg_context_unsqueezed
        ), dim=2)

        output, hidden = self.rnn(rnn_input, decoder_hidden)
        prediction = self.fc_out(output.squeeze(0))

        return prediction, hidden, context.squeeze(1)

In [6]:
# ==================================================================================
# DIVERSITY LOSS (Unchanged)
# ==================================================================================
class DiversityLoss(nn.Module):
    """Encourages diverse token predictions to prevent mode collapse"""
    def __init__(self, vocab_size):
        super().__init__()
        self.vocab_size = vocab_size
    
    def forward(self, logits):
        # logits: [batch, seq_len, vocab_size]
        batch_size, seq_len, vocab_size = logits.shape
        
        # Get probabilities
        probs = F.softmax(logits, dim=-1)
        
        # Average probability distribution across batch and sequence
        avg_probs = probs.mean(dim=(0, 1))  # [vocab_size]
        
        # Uniform distribution
        uniform = torch.ones_like(avg_probs) / self.vocab_size
        
        # KL divergence from uniform (lower = more diverse)
        kl_div = F.kl_div(avg_probs.log(), uniform, reduction='batchmean')
        
        return kl_div

In [7]:
# ==================================================================================
# UPDATED: UNCERTAINTY-WEIGHTED LOSS (Removed Category)
# ==================================================================================
class UncertaintyWeightedLoss(nn.Module):
    """Learns optimal task weighting automatically"""
    def __init__(self):
        super().__init__()
        # Log variance parameters (learnable)
        self.log_var_text = nn.Parameter(torch.zeros(1))
        self.log_var_color = nn.Parameter(torch.zeros(1))
        self.log_var_object = nn.Parameter(torch.zeros(1))
        # Removed category
    
    def forward(self, loss_t, loss_c, loss_o):
        # Precision = exp(-log_var)
        precision_t = torch.exp(-self.log_var_text)
        precision_c = torch.exp(-self.log_var_color)
        precision_o = torch.exp(-self.log_var_object)
        
        # Weighted loss + regularization
        loss = (precision_t * loss_t + self.log_var_text +
                precision_c * loss_c + self.log_var_color +
                precision_o * loss_o + self.log_var_object)
        
        return loss

In [8]:
# ==================================================================================
# MAIN MODEL
# ==================================================================================
class Seq2Seq(nn.Module):
    def __init__(self, text_vocab_size, num_colors, num_objects, enc_hidden=256, dec_hidden=256,
                 pad_id=0, dropout=0.2, color_feature_dim=32, object_feature_dim=32, emb_dim=256, dec_layers=2):
        super().__init__()
        self.encoder = SpatioTemporalEEGEncoder(enc_hidden=enc_hidden, dropout=dropout, num_layers=dec_layers)
        
        # Updated MetadataEncoder
        self.meta_encoder = MetadataEncoder(num_colors, num_objects,
                                            color_feature_dim, object_feature_dim)

        meta_features_dim = self.meta_encoder.output_dim
        enc_dim = enc_hidden * 2

        self.decoder = Decoder(text_vocab_size, emb_dim, enc_hidden, dec_hidden,
                               meta_features_dim, dec_layers, pad_id, dropout)

        # Updated Meta Head (No Categories)
        self.meta_head = nn.Sequential(
            nn.Linear(enc_dim, 256),
            nn.ReLU(),
            nn.LayerNorm(256),
            nn.Dropout(0.3),
            nn.Linear(256, num_colors + num_objects)
        )
        self.num_colors = num_colors
        self.num_objects = num_objects

    def forward(self, eeg, metadata, target_text, edge_index, edge_attr, teacher_forcing_ratio=0.5):
        batch_size = eeg.shape[0]
        target_len = target_text.shape[1]
        target_vocab_size = self.decoder.vocab_size

        encoder_outputs, encoder_hidden = self.encoder(eeg, edge_index, edge_attr)
        meta_features = self.meta_encoder(metadata)

        decoder_hidden = self.decoder.init_hidden(encoder_hidden)

        hidden_reshaped = encoder_hidden.view(self.encoder.rnn.num_layers, 2, batch_size, -1)
        last_layer_hidden = hidden_reshaped[-1]
        global_eeg_context = torch.cat((last_layer_hidden[0], last_layer_hidden[1]), dim=1)

        meta_preds = self.meta_head(global_eeg_context)
        pred_color = meta_preds[:, :self.num_colors]
        pred_object = meta_preds[:, self.num_colors:]

        outputs = torch.zeros(target_len, batch_size, target_vocab_size).to(eeg.device)
        decoder_input = target_text[:, 0]

        for t in range(1, target_len):
            output, decoder_hidden, _ = self.decoder(
                decoder_input,
                decoder_hidden,
                encoder_outputs,
                meta_features,
                global_eeg_context
            )

            outputs[t] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            decoder_input = target_text[:, t] if teacher_force else top1

        return outputs[1:].permute(1, 0, 2), pred_color, pred_object

In [9]:
# ==================================================================================
# TRAINING AND EVALUATION
# ==================================================================================

def train_one_epoch(model, loader, optimizer, text_criterion, color_criterion,
                   object_criterion, diversity_criterion, uncertainty_loss, granger_edge_index, 
                   granger_edge_attr, diversity_weight=0.01):
    model.train()
    total_loss = 0.0
    total_loss_components = {'text': 0.0, 'color': 0.0, 'object': 0.0, 'diversity': 0.0}
    
    progress_bar = tqdm(loader, desc="Training", leave=False)

    for eeg_b, meta_b, txt_b in progress_bar:
        eeg_b, txt_b, meta_b = eeg_b.to(device), txt_b.to(device), meta_b.to(device)

        optimizer.zero_grad()

        text_logits, pred_color, pred_object = model(
            eeg_b, meta_b, txt_b, granger_edge_index, granger_edge_attr, teacher_forcing_ratio=0.5
        )

        # Individual losses (Note: Color is now Multi-Label/BCE)
        loss_t = text_criterion(text_logits.reshape(-1, text_logits.shape[-1]), txt_b[:, 1:].reshape(-1))
        loss_c = color_criterion(pred_color, meta_b[:, :NUM_COLORS].float())
        loss_o = object_criterion(pred_object, meta_b[:, NUM_COLORS:].float())
        
        # Diversity loss
        loss_div = diversity_criterion(text_logits)

        # Uncertainty-weighted combination (Removed category)
        loss_main = uncertainty_loss(loss_t, loss_c, loss_o)
        
        # Total loss
        loss = loss_main + diversity_weight * loss_div

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        total_loss_components['text'] += loss_t.item()
        total_loss_components['color'] += loss_c.item()
        total_loss_components['object'] += loss_o.item()
        total_loss_components['diversity'] += loss_div.item()

        progress_bar.set_postfix(
            loss=loss.item(), 
            txt=loss_t.item(), 
            div=loss_div.item()
        )

    n = len(loader)
    return {k: v/n for k, v in total_loss_components.items()}, total_loss / n


@torch.no_grad()
def evaluate(model, loader, text_criterion, color_criterion, object_criterion,
             diversity_criterion, uncertainty_loss, granger_edge_index, granger_edge_attr, diversity_weight=0.01):
    model.eval()
    total_loss = 0.0
    total_loss_components = {'text': 0.0, 'color': 0.0, 'object': 0.0, 'diversity': 0.0}
    
    progress_bar = tqdm(loader, desc="Evaluating", leave=False)

    for eeg_b, meta_b, txt_b in progress_bar:
        eeg_b, txt_b, meta_b = eeg_b.to(device), txt_b.to(device), meta_b.to(device)

        text_logits, pred_color, pred_object = model(
            eeg_b, meta_b, txt_b, granger_edge_index, granger_edge_attr, teacher_forcing_ratio=0.0
        )

        loss_t = text_criterion(text_logits.reshape(-1, text_logits.shape[-1]), txt_b[:, 1:].reshape(-1))
        loss_c = color_criterion(pred_color, meta_b[:, :NUM_COLORS].float())
        loss_o = object_criterion(pred_object, meta_b[:, NUM_COLORS:].float())
        loss_div = diversity_criterion(text_logits)

        loss_main = uncertainty_loss(loss_t, loss_c, loss_o)
        loss = loss_main + diversity_weight * loss_div

        total_loss += loss.item()
        total_loss_components['text'] += loss_t.item()
        total_loss_components['color'] += loss_c.item()
        total_loss_components['object'] += loss_o.item()
        total_loss_components['diversity'] += loss_div.item()

        progress_bar.set_postfix(loss=loss.item())

    n = len(loader)
    return {k: v/n for k, v in total_loss_components.items()}, total_loss / n

In [10]:
# ==================================================================================
# INFERENCE WITH CONTEXT BOOSTING
# ==================================================================================

@torch.no_grad()
def generate_with_context_boost(model, eeg_signal, meta_signal, edge_index, edge_attr,
                                k=5, penalty_alpha=0.3, context_beta=0.7, max_len=100):
    model.eval()
    eeg_signal = eeg_signal.unsqueeze(0).to(device)
    meta_signal = meta_signal.unsqueeze(0).to(device)

    encoder_outputs, encoder_hidden = model.encoder(eeg_signal, edge_index, edge_attr)
    meta_features = model.meta_encoder(meta_signal)

    decoder_hidden = model.decoder.init_hidden(encoder_hidden)

    hidden_reshaped = encoder_hidden.view(model.encoder.rnn.num_layers, 2, 1, -1)
    last_layer_hidden = hidden_reshaped[-1]
    global_eeg_context = torch.cat((last_layer_hidden[0], last_layer_hidden[1]), dim=1)

    meta_preds = model.meta_head(global_eeg_context)
    
    # Updated for Multi-Label Inference
    pred_color_ids = (torch.sigmoid(meta_preds[0, :model.num_colors]) > 0.5).nonzero(as_tuple=True)[0].tolist()
    pred_object_ids = (torch.sigmoid(meta_preds[0, model.num_colors:]) > 0.5).nonzero(as_tuple=True)[0].tolist()

    generated_ids = torch.tensor([SOS_ID], device=device)

    for step in range(max_len):
        input_token = generated_ids[-1].unsqueeze(0)

        prediction, new_hidden, attention_context = model.decoder(
            input_token,
            decoder_hidden,
            encoder_outputs,
            meta_features,
            global_eeg_context
        )

        decoder_hidden = new_hidden
        model_log_probs = F.log_softmax(prediction, dim=-1).squeeze(0)
        
        current_seq_len = generated_ids.shape[0]
        prev_token_embeddings = F.normalize(model.decoder.embedding(generated_ids), dim=-1)
        
        topk_model_log_probs, topk_ids = torch.topk(model_log_probs, k)
        candidate_token_embeddings = F.normalize(model.decoder.embedding(topk_ids), dim=-1)
        
        sim_matrix = torch.matmul(candidate_token_embeddings, prev_token_embeddings.t())
        degeneration_penalty = torch.zeros(k, device=device)
        if current_seq_len > 1:
            degeneration_penalty, _ = torch.max(sim_matrix, dim=-1)
            
        current_decoder_state = F.normalize(decoder_hidden[-1].squeeze(), dim=-1)
        context_agreement_score = torch.matmul(candidate_token_embeddings, current_decoder_state)
        
        final_score = topk_model_log_probs + context_beta * context_agreement_score - penalty_alpha * degeneration_penalty
        
        best_next_token_idx = torch.argmax(final_score)
        next_token_id = topk_ids[best_next_token_idx]

        generated_ids = torch.cat([generated_ids, next_token_id.unsqueeze(0)])
        if next_token_id.item() == EOS_ID:
            break
            
    if generated_ids.numel() > 1:
        predicted_text_ids = generated_ids[1:-1] if generated_ids[-1].item() == EOS_ID else generated_ids[1:]
        predicted_text = tokenizer.decode(predicted_text_ids.tolist(), skip_special_tokens=True)
    else:
        predicted_text = ""
    
    return predicted_text, pred_color_ids, pred_object_ids

In [11]:
# ==================================================================================
# MAIN EXECUTION: Initialization
# ==================================================================================
print("--- Starting Initialization ---")
# Create dataset and loaders
dataset = EEGMetaTextH5Dataset(H5_FILE_PATH)
N = len(dataset)
n_train = int(N * TRAIN_PCT)
n_val = int(N * VAL_PCT)
n_test = N - n_train - n_val
g = torch.Generator().manual_seed(42)
train_ds, val_ds, test_ds = random_split(dataset, [n_train, n_val, n_test], generator=g)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_multimodal_batch)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_multimodal_batch)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_multimodal_batch)

# Create Granger matrix
print("Creating Granger Causality matrix...")
try:
    eeg_b, _, _ = next(iter(train_loader))
    granger_edge_index, granger_edge_attr = create_granger_causality_matrix(eeg_b)
    num_channels = eeg_b.shape[1]
    granger_edge_index, granger_edge_attr = add_self_loops(
        granger_edge_index, edge_attr=granger_edge_attr, num_nodes=num_channels, fill_value=1.0
    )
    if granger_edge_attr is None:
        granger_edge_attr = torch.ones(granger_edge_index.shape[1], dtype=torch.float)
    granger_edge_index = granger_edge_index.to(torch.long).to(device)
    granger_edge_attr = granger_edge_attr.to(torch.float32).to(device)
    print(f"Granger matrix created: {granger_edge_index.shape}")
except Exception as e:
    print(f"Error creating Granger matrix: {e}. Using fallback.")
    num_channels = 62
    edge_index = torch.combinations(torch.arange(num_channels), r=2).t().contiguous()
    edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)
    edge_index, _ = add_self_loops(edge_index, num_nodes=num_channels)
    granger_edge_index = edge_index.to(torch.long).to(device)
    granger_edge_attr = torch.ones(granger_edge_index.shape[1], dtype=torch.float32).to(device)

# Instantiate model
model = Seq2Seq(
    text_vocab_size=TEXT_VOCAB_SIZE,
    num_colors=NUM_COLORS,
    num_objects=NUM_OBJECTS,
    pad_id=PAD_ID,
    dropout=0.2,
    enc_hidden=256,
    dec_hidden=256,
    emb_dim=256,
    dec_layers=2
).to(device)

print(f"Model instantiated on '{device}'.")
print(f"Total parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# --- WEIGHTS FOR NEW DATASET (1400 SAMPLES) ---
object_pos_weight = torch.tensor([3.1176, 5.6667, 6.1066, 1.1021, 2.0905, 7.5366]).to(device)
color_pos_weight = torch.tensor([3.1543, 1.2764, 3.9645, 1.7888, 0.8301, 11.2807, 5.2780, 1.0408, 4.4054]).to(device)

# Loss functions
text_criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
color_criterion = nn.BCEWithLogitsLoss(pos_weight=color_pos_weight)
object_criterion = nn.BCEWithLogitsLoss(pos_weight=object_pos_weight)
diversity_criterion = DiversityLoss(TEXT_VOCAB_SIZE).to(device)
uncertainty_loss = UncertaintyWeightedLoss().to(device)

# Optimizer includes uncertainty loss parameters
optimizer = AdamW(list(model.parameters()) + list(uncertainty_loss.parameters()), 
                  lr=3e-5, weight_decay=1e-2)
scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.2, patience=2, verbose=True)

# Training hyperparameters (must be initialized outside the loop)
DIVERSITY_WEIGHT = 0.01
best_val_loss = float('inf')

print("\n--- Initialization Complete. Ready for Training. ---")

--- Starting Initialization ---
Creating Granger Causality matrix...


/home/poorna/venvs/torch/lib64/python3.11/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(


Granger matrix created: torch.Size([2, 2288])
Encoder RNN input size: 256
MetadataEncoder output dimension: 64
Decoder RNN input dimension: 1344
Model instantiated on 'cuda'.
Total parameters: 19,740,617

--- Initialization Complete. Ready for Training. ---


/home/poorna/venvs/torch/lib64/python3.11/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [16]:
print("\n--- Starting Training ---")
for epoch in range(1, EPOCHS + 1):
    start_time = time.time()

    train_components, train_loss = train_one_epoch(
        model, train_loader, optimizer,
        text_criterion, color_criterion, object_criterion,
        diversity_criterion, uncertainty_loss,
        granger_edge_index, granger_edge_attr, DIVERSITY_WEIGHT
    )
    
    val_components, val_loss = evaluate(
        model, val_loader,
        text_criterion, color_criterion, object_criterion,
        diversity_criterion, uncertainty_loss,
        granger_edge_index, granger_edge_attr, DIVERSITY_WEIGHT
    )

    scheduler.step(val_loss)
    end_time = time.time()
    
    print(f'\nEpoch: {epoch:02} | Time: {int(end_time - start_time)}s')
    print(f'\tTrain Loss: {train_loss:.4f} | Text: {train_components["text"]:.4f} | Div: {train_components["diversity"]:.4f}')
    print(f'\t  Val Loss: {val_loss:.4f} | Text: {val_components["text"]:.4f} | Div: {val_components["diversity"]:.4f}')
    
    with torch.no_grad():
        print(f'\tTask Weights: Text={torch.exp(-uncertainty_loss.log_var_text).item():.3f}, '
              f'Color={torch.exp(-uncertainty_loss.log_var_color).item():.3f}, '
              f'Obj={torch.exp(-uncertainty_loss.log_var_object).item():.3f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'eeg-text-enhanced-model.pt')
        print(f"\t-> Val loss decreased. Saved model.")

    # After epoch ends, check for collapse (Corrected unpack)
    if epoch % 5 == 0:
        print("Checking for mode collapse")
        sample_outputs = []
        for j in range(10):
            # CORRECTED: generate_with_context_boost returns 3 values, so we unpack 3
            pred, _, _ = generate_with_context_boost( 
                model, test_ds[j][0], test_ds[j][1], 
                granger_edge_index, granger_edge_attr
            )
            sample_outputs.append(pred)
        
        unique_rate = len(set(sample_outputs)) / len(sample_outputs)
        print(f"  Diversity check: {unique_rate*100:.1f}% unique outputs")
        print(f"Sample predictions: {sample_outputs}")
        
        if unique_rate < 0.5:
            print("  ⚠️ Mode collapse detected! Increasing diversity weight")
            # Note: DIVERSITY_WEIGHT is a global variable from Block 1, which this loop modifies.
            DIVERSITY_WEIGHT *= 1.5

print("\n--- Training Complete ---")


--- Starting Training ---


Training:   0%|          | 0/1400 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/175 [00:00<?, ?it/s]


Epoch: 01 | Time: 470s
	Train Loss: 5.0696 | Text: 3.0904 | Div: 0.0002
	  Val Loss: 5.6946 | Text: 3.7715 | Div: 0.0002
	Task Weights: Text=0.960, Color=1.013, Obj=0.983
	-> Val loss decreased. Saved model.


Training:   0%|          | 0/1400 [00:00<?, ?it/s]

...

Training:   0%|          | 0/1400 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/175 [00:00<?, ?it/s]


Epoch: 23 | Time: 472s
	Train Loss: 3.5326 | Text: 1.7606 | Div: 0.0003
	  Val Loss: 4.4489 | Text: 3.2459 | Div: 0.0004
	Task Weights: Text=0.555, Color=1.048, Obj=0.981
	-> Val loss decreased. Saved model.


Training:   0%|          | 0/1400 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [12]:
# Assuming these lists are available from the initial setup context:
COLOR_NAMES = ["Black", "Blue", "Brown", "Green", "Grey", "Orange", "Red", "White", "Yellow"]
OBJECT_NAMES = ["Animal", "Building", "Food", "Nature", "Person", "Vehicle"]

# ==============================================================================
# INFERENCE (with Ground Truth Metadata)
# ==============================================================================
print("\n--- Starting Inference and Evaluation ---")

# Load best model
try:
    model.load_state_dict(torch.load('eeg-text-enhanced-model.pt', map_location=device))
    print(f"Best model loaded for inference.")
except FileNotFoundError:
    print("Warning: Model checkpoint 'eeg-text-enhanced-model.pt' not found. Using current model state.")

# Load object mapping (for predicted object names)
try:
    with open(OBJECT_MAPPING_FILE, 'r') as f:
        object_mapping = json.load(f)
    print(f"Object mapping loaded: {len(object_mapping)} objects")
except FileNotFoundError:
    print(f"Warning: '{OBJECT_MAPPING_FILE}' not found. Metadata prediction names will use IDs.")
    object_mapping = {}

print(f"\n--- Running Inference ---")
predictions = []
references = []
    
NUM_SAMPLES = min(len(test_ds), 20)
for i in range(NUM_SAMPLES):
    eeg_sample, meta_sample, true_text_ids = test_ds[i]
    
    # 1. Get Ground Truth Metadata Names
    gt_meta_vector = meta_sample.cpu().numpy()
    gt_color_ids = np.where(gt_meta_vector[:NUM_COLORS] == 1)[0]
    gt_object_ids = np.where(gt_meta_vector[NUM_COLORS:] == 1)[0]
    
    gt_color_names = [COLOR_NAMES[idx] for idx in gt_color_ids]
    gt_object_names = [OBJECT_NAMES[idx] for idx in gt_object_ids]

    # 2. Get Predictions (Corrected unpack)
    predicted_text, pred_color_ids, pred_object_ids = generate_with_context_boost(
        model, eeg_sample, meta_sample, granger_edge_index, granger_edge_attr
    )

    true_text = tokenizer.decode(true_text_ids.tolist(), skip_special_tokens=True)
    predictions.append(predicted_text)
    references.append(true_text)

    # 3. Convert Predicted Object IDs to names
    # Note: The provided OBJECT_NAMES list is for the 6 high-level categories.
    # The 'object_mapping' file might contain finer-grained object names from the dataset.
    # We will use the 6 high-level names for consistency, assuming the indices match the original setup.
    pred_color_names = [COLOR_NAMES[oid] for oid in pred_color_ids if oid < NUM_COLORS]
    pred_object_names = [OBJECT_NAMES[oid] for oid in pred_object_ids if oid < NUM_OBJECTS]
    
    # Display results
    print(f"\n--- Sample {i+1}/{NUM_SAMPLES} ---")
    print(f"GT Text:      {true_text}")
    print(f"GT Colors:    {gt_color_names}")
    print(f"GT Objects:   {gt_object_names}")
    print(f"Pred Text:    {predicted_text}")
    print(f"Pred Colors:  {pred_color_names}")
    print(f"Pred Objs:    {pred_object_names}")

# Metrics
try:
    bleu_metric = hf_evaluate.load('bleu')
    bleu_results = bleu_metric.compute(predictions=predictions, references=[[r] for r in references])
    print(f"\n=== FINAL RESULTS ===")
    print(f"BLEU Score: {bleu_results['bleu']:.4f}")
    
    rouge_metric = hf_evaluate.load('rouge')
    rouge_results = rouge_metric.compute(predictions=predictions, references=references)
    print(f"ROUGE-1: {rouge_results['rouge1']:.4f} | ROUGE-L: {rouge_results['rougeL']:.4f}")
except Exception as e:
    print(f"Error computing metrics: {e}")

print("\n--- Inference Complete ---")


--- Starting Inference and Evaluation ---
Best model loaded for inference.
Object mapping loaded: 6 objects

--- Running Inference ---

--- Sample 1/20 ---
GT Text:      a group of yellow fish swimming in the ocean
GT Colors:    ['Blue', 'Green', 'Orange']
GT Objects:   ['Animal', 'Nature']
Pred Text:    a turtle swimming in the ocean
Pred Colors:  ['Brown', 'Green', 'Grey', 'Orange']
Pred Objs:    ['Animal', 'Nature']

--- Sample 2/20 ---
GT Text:      a river with a waterfall in the middle
GT Colors:    ['Blue', 'Green', 'White']
GT Objects:   ['Nature']
Pred Text:    a mountain with a lot of rocks and trees
Pred Colors:  ['Brown', 'Orange', 'White', 'Yellow']
Pred Objs:    ['Animal', 'Food']

--- Sample 3/20 ---
GT Text:      fireworks in the dark sky
GT Colors:    ['Blue', 'White', 'Yellow']
GT Objects:   []
Pred Text:    a jellyfish in the water
Pred Colors:  ['Black', 'Blue', 'Brown', 'Green', 'White']
Pred Objs:    ['Animal', 'Food', 'Nature']

--- Sample 4/20 ---
GT Text:     